# Next-Activity Prediction on CASAS Smart-Home Data
**DA331 Big Data Analytics, Assignment 5 (Task 2)**

**Goal.** Given the resident's activity history up to now, predict the **next activity** they will perform.

**Pipeline**
1. Load raw sensor events (same loader as the EDA notebook)
2. Turn events into **activity episodes** (runs of one label; `Other_Activity` removed, fragments merged)
3. Build **temporal features** for each episode *t*: the current and previous activities (lags),
   time of day (cyclic), day of week, duration, number of events, last sensor, time since the previous episode
4. **Chronological split per home**: first 70% train, next 15% validation, last 15% test (no future leakage)
5. Models, from simplest up:
   majority class, 1st-order Markov chain, 2nd-order Markov chain, time-aware Markov chain,
   and a **HistGradientBoosting** classifier on the temporal features
6. Evaluation: accuracy, macro-F1, top-3 accuracy, per-class report, confusion matrix, feature importance
7. Save the model and demo inference

In [ ]:
import os, re, gzip, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

# ---- Configuration ---------------------------------------------------------
# On Kaggle: "Add Input" -> the CASAS dataset; it is mounted under /kaggle/input.
DATA_DIR = Path(os.environ.get("CASAS_DATA_DIR",
                "/kaggle/input" if Path("/kaggle/input").exists() else "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_FILES = None                 # e.g. 5 to prototype on a few homes first
OTHER_LABELS = {"Other_Activity", "Other", "other", "None", "none", "NULL", ""}
print("DATA_DIR =", DATA_DIR, "| OUT_DIR =", OUT_DIR)

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
import joblib
RANDOM_STATE = 42
N_LAGS = 3                 # how many previous activities to use as features
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
MIN_CLASS_EPISODES = 20    # rarer activities are grouped as "Rare_Activity"

## Data loading

CASAS smart-home files are plain-text event logs, one sensor event per line:

```
<date> <time>  <sensor id> [<location/alias>]  <message>  [<activity label>]
2012-07-20 12:51:19.270391   BedroomAArea   ON   Sleep
```

Different releases (and the Kaggle mirror) differ slightly: tab vs space vs comma separated,
with/without a header, an optional extra location column, gz-compressed, and older releases
mark activities only as `Activity begin` / `Activity end`. The loader below auto-detects all
of these, uses **vectorised regex parsing** (no Python loop over rows), stores strings as
**categoricals** to keep memory small, and caches the result as **Parquet** so that re-runs are instant.

In [ ]:
STATE_RE = r"ON|OFF|OPEN|CLOSE|OPENED|CLOSED|PRESENT|ABSENT|TRUE|FALSE"
EVENT_RE = re.compile(  # date, time, rest-of-line
    r"^(?P<date>\d{4}-\d{2}-\d{2})[ T]+(?P<time>\d{1,2}:\d{2}(?::\d{2}(?:\.\d+)?)?)\S*\s+(?P<rest>.+)$")
REST_STATE_RE = (  # sensor [alias...] STATE [label...]
    rf"^(?P<sensor>\S+)(?:\s+(?P<location>\S+(?:\s+\S+)*?))?\s+(?P<message>{STATE_RE})(?:\s+(?P<activity>.*))?$")
REST_GENERIC_RE = r"^(?P<sensor>\S+)\s+(?P<message>\S+)(?:\s+(?P<activity>.*))?$"   # e.g. T001 21.5

COLUMN_ALIASES = {
    "timestamp": ["timestamp", "datetime", "date_time", "time_stamp", "ts"],
    "date": ["date", "day"], "time": ["time"],
    "sensor": ["sensor", "sensor_id", "sensorid", "sensor_name", "device", "item"],
    "location": ["location", "sensor_location", "alias", "room", "sensor_alias"],
    "message": ["message", "state", "value", "status", "sensor_state", "sensor_status", "event", "reading"],
    "activity": ["activity", "label", "activity_label", "annotation", "class", "activity_name"],
}
DATA_EXT = {".txt", ".csv", ".tsv", ".dat", ".gz", ".parquet", ""}
SKIP_NAME = re.compile(r"readme|license|licence|description|metadata|\.md$|\.json$|\.pdf$|\.py$|\.ipynb$", re.I)
GENERIC_STEMS = {"data", "ann", "annotated", "raw", "rawdata", "events", "labels", "sensor", "sensors"}


def find_data_files(root):
    files = [p for p in Path(root).rglob("*")
             if p.is_file() and p.suffix.lower() in DATA_EXT and not SKIP_NAME.search(p.name)
             and p.stat().st_size > 0 and "/kaggle/working" not in str(p)]
    return sorted(files)


def home_name(p):
    stem = p.name.split(".")[0]
    return p.parent.name if stem.lower() in GENERIC_STEMS else stem


def _open_text(p):
    return gzip.open(p, "rt", errors="replace") if p.suffix.lower() == ".gz" else open(p, "r", errors="replace")


def _map_columns(cols):
    low = {c: re.sub(r"[^a-z_]", "", c.strip().lower().replace(" ", "_")) for c in cols}
    out = {}
    for std, names in COLUMN_ALIASES.items():
        for c, l in low.items():
            if l in names and c not in out.values():
                out[std] = c
                break
    return out


def _from_table(df):
    """Normalise a table that has a header (CSV / Parquet) to the standard schema."""
    m = _map_columns(df.columns)
    if "timestamp" in m:
        ts = df[m["timestamp"]].astype(str)
    elif "date" in m and "time" in m:
        ts = df[m["date"]].astype(str) + " " + df[m["time"]].astype(str)
    else:
        raise ValueError(f"Cannot find timestamp columns in {list(df.columns)}")
    if "sensor" not in m or "message" not in m:
        raise ValueError(f"Cannot find sensor/message columns in {list(df.columns)}")
    out = pd.DataFrame({"timestamp": ts, "sensor": df[m["sensor"]].astype(str),
                        "message": df[m["message"]].astype(str)})
    out["location"] = df[m["location"]].astype(str) if "location" in m else np.nan
    out["activity"] = df[m["activity"]].astype(str) if "activity" in m else np.nan
    return out


def _from_lines(lines):
    """Vectorised parse of header-less CASAS text lines."""
    s = pd.Series(lines, dtype="string").str.replace(r"[,\t;]+", " ", regex=True).str.strip()
    ev = s.str.extract(EVENT_RE)
    ev = ev[ev["date"].notna()]
    rest = ev["rest"]
    parsed = rest.str.extract(REST_STATE_RE, flags=re.I)
    miss = parsed["sensor"].isna()
    if miss.any():
        gen = rest[miss].str.extract(REST_GENERIC_RE)
        parsed.loc[miss, ["sensor", "message", "activity"]] = gen[["sensor", "message", "activity"]].values
    return pd.DataFrame({"timestamp": ev["date"] + " " + ev["time"].fillna("00:00:00"),
                         "sensor": parsed["sensor"], "location": parsed["location"],
                         "message": parsed["message"].str.upper(), "activity": parsed["activity"]})


def _expand_begin_end(df):
    """Old CASAS format: labels only as 'X begin' / 'X end' -> label every event in between."""
    act = df["activity"].fillna("").astype(str).str.strip()
    mark = act.str.extract(r"^(?P<name>.+?)\s+(?P<kind>begin|end)$", flags=re.I)
    if mark["kind"].notna().sum() == 0 or (act[mark["kind"].isna()] != "").mean() > 0.01:
        return df                                   # already one label per event
    lab = pd.Series(np.nan, index=df.index, dtype=object)     # label of the marker row itself
    after = pd.Series(np.nan, index=df.index, dtype=object)   # what is active after the marker
    active = []
    for idx, name, kind in mark.dropna().itertuples():
        if kind.lower() == "begin":
            active.append(name)
        elif name in active:
            active.remove(name)
        lab[idx] = name
        after[idx] = active[-1] if active else "Other_Activity"
    cur = lab.fillna(after.shift(1).ffill())
    df = df.copy()
    df["activity"] = cur.ffill().fillna("Other_Activity")
    return df


def load_file(p):
    if p.suffix.lower() == ".parquet":
        df = _from_table(pd.read_parquet(p))
    else:
        with _open_text(p) as f:
            head = [f.readline() for _ in range(5)]
        first = next((l for l in head if l.strip()), "")
        has_header = bool(first) and not re.match(r"^\s*\d{4}-\d{2}-\d{2}", first) and bool(
            re.search(r"sensor|time|date|activity|label|message|state", first, re.I))
        if has_header:
            sep = max([",", "\t", ";"], key=first.count)
            df = _from_table(pd.read_csv(p, sep=sep, dtype=str, compression="infer"))
        else:
            with _open_text(p) as f:
                df = _from_lines(f.read().splitlines())
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", errors="coerce")
    df = df.dropna(subset=["timestamp", "sensor"])
    df["activity"] = df["activity"].where(df["activity"].notna(), np.nan)
    if df["activity"].notna().any():
        df["activity"] = df["activity"].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})
        df = _expand_begin_end(df)
    df.insert(0, "home", home_name(p))
    return df


def sensor_type(msg):
    m = msg.astype(str).str.upper()
    num = pd.to_numeric(m, errors="coerce").notna()
    return np.select([m.isin(["ON", "OFF"]), m.isin(["OPEN", "CLOSE", "OPENED", "CLOSED"]), num],
                     ["motion", "door", "numeric"], "other")


def load_casas(data_dir, max_files=None, cache=None):
    if cache is not None and Path(cache).exists():
        print("Loading cached", cache)
        return pd.read_parquet(cache)
    files = find_data_files(data_dir)[:max_files]
    if not files:
        raise FileNotFoundError(f"No data files found under {data_dir}. Attach the dataset (Kaggle: Add Input).")
    parts, t0 = [], time.time()
    for p in files:
        try:
            d = load_file(p)
        except Exception as e:                      # keep going if one odd file is present
            print(f"  skip {p.name}: {e}"); continue
        if len(d) == 0:
            print(f"  skip {p.name}: no parsable events"); continue
        parts.append(d)
        print(f"  {p.relative_to(data_dir)}: {len(d):,} events")
    df = pd.concat(parts, ignore_index=True)
    df = df.sort_values(["home", "timestamp"], kind="stable").reset_index(drop=True)
    df["sensor_type"] = sensor_type(df["message"])
    for c in ["home", "sensor", "location", "message", "activity", "sensor_type"]:
        df[c] = df[c].astype("category")
    print(f"Loaded {len(df):,} events from {df['home'].nunique()} home(s) in {time.time()-t0:.1f}s")
    if cache is not None:
        try:
            df.to_parquet(cache, index=False)
        except Exception as e:
            print(f"Parquet cache not written ({type(e).__name__}); continuing without cache")
    return df

In [ ]:
df = load_casas(DATA_DIR, max_files=MAX_FILES, cache=OUT_DIR / f"casas_events_{MAX_FILES or 'all'}.parquet")
print(df.shape); df.head()

## 1. Events to activity episodes

In [ ]:
def build_episodes(df, drop_other=True, merge_gap_min=10):
    """Collapse the event stream into activity episodes (runs of the same label).

    If drop_other, 'Other_Activity' runs are removed and neighbouring runs of the same
    activity that are closer than merge_gap_min minutes are merged back together
    (e.g. Sleep - a few unlabelled events - Sleep  ->  one Sleep episode).
    """
    d = df.loc[df["activity"].notna(), ["home", "timestamp", "activity", "sensor"]].copy()
    for c in ["home", "activity", "sensor"]:
        d[c] = d[c].astype(str)
    new = (d["activity"] != d["activity"].shift()) | (d["home"] != d["home"].shift())
    ep = d.groupby(new.cumsum(), sort=False).agg(
        home=("home", "first"), activity=("activity", "first"),
        start=("timestamp", "first"), end=("timestamp", "last"),
        n_events=("timestamp", "size"), last_sensor=("sensor", "last"),
        n_sensors=("sensor", "nunique")).reset_index(drop=True)
    if drop_other:
        ep = ep[~ep["activity"].isin(OTHER_LABELS)].reset_index(drop=True)
        gap = (ep["start"] - ep["end"].shift()).dt.total_seconds() / 60
        new = ((ep["activity"] != ep["activity"].shift()) | (ep["home"] != ep["home"].shift())
               | (gap > merge_gap_min))
        ep = ep.groupby(new.cumsum(), sort=False).agg(
            home=("home", "first"), activity=("activity", "first"), start=("start", "min"),
            end=("end", "max"), n_events=("n_events", "sum"), last_sensor=("last_sensor", "last"),
            n_sensors=("n_sensors", "max")).reset_index(drop=True)
    ep["duration_min"] = (ep["end"] - ep["start"]).dt.total_seconds() / 60
    return ep

In [ ]:
ep = build_episodes(df, drop_other=True, merge_gap_min=10)
counts = ep["activity"].value_counts()
rare = counts[counts < MIN_CLASS_EPISODES].index
ep["activity"] = ep["activity"].where(~ep["activity"].isin(rare), "Rare_Activity")
print(f"{len(ep):,} episodes, {ep['activity'].nunique()} classes ({len(rare)} rare labels grouped)")
ep.head()

## 2. Temporal feature engineering
For each episode *t* in a home, the **target** is the activity of episode *t+1*. Every feature only uses
information available when episode *t* ends, so nothing is taken from the future.

In [ ]:
def make_features(ep, n_lags=N_LAGS):
    ep = ep.sort_values(["home", "start"]).reset_index(drop=True)
    g = ep.groupby("home", sort=False)
    X = pd.DataFrame(index=ep.index)
    X["cur_act"] = ep["activity"]
    for k in range(1, n_lags + 1):
        X[f"prev{k}_act"] = g["activity"].shift(k).fillna("<START>")
    h = ep["end"].dt.hour + ep["end"].dt.minute / 60
    X["hour"] = h
    X["hour_sin"], X["hour_cos"] = np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24)
    X["dow"] = ep["end"].dt.dayofweek
    X["is_weekend"] = (X["dow"] >= 5).astype(int)
    X["log_duration"] = np.log1p(ep["duration_min"])
    X["log_n_events"] = np.log1p(ep["n_events"])
    X["n_sensors"] = ep["n_sensors"]
    X["last_sensor"] = ep["last_sensor"]
    X["log_gap_prev"] = np.log1p(((ep["start"] - g["end"].shift()).dt.total_seconds() / 60).clip(lower=0)).fillna(0)
    y = g["activity"].shift(-1)
    meta = ep[["home", "start", "end"]].copy()
    keep = y.notna()
    return X[keep].reset_index(drop=True), y[keep].reset_index(drop=True), meta[keep].reset_index(drop=True)

X, y, meta = make_features(ep)
print(X.shape); X.head()

In [ ]:
# Chronological split inside every home (70/15/15), so the test set is always the most recent period.
pos = meta.groupby("home").cumcount() / meta.groupby("home")["home"].transform("size")
split = np.where(pos < TRAIN_FRAC, "train", np.where(pos < TRAIN_FRAC + VAL_FRAC, "val", "test"))
idx = {s: np.where(split == s)[0] for s in ["train", "val", "test"]}
for s, i in idx.items():
    print(f"{s:5s}: {len(i):6,} samples  {meta.loc[i, 'start'].min():%Y-%m-%d} to {meta.loc[i, 'end'].max():%Y-%m-%d}")

CLASSES = np.array(sorted(y.unique()))
cls_id = {c: i for i, c in enumerate(CLASSES)}
y_all = y.map(cls_id).to_numpy()
ytr, yva, yte = (y_all[idx[s]] for s in ["train", "val", "test"])

## 3. Baselines: majority class and Markov chains

In [ ]:
def topk_acc(P, y_true, k=3):
    k = min(k, P.shape[1])
    topk = np.argsort(-P, axis=1)[:, :k]
    return float(np.mean([t in row for t, row in zip(y_true, topk)]))

def evaluate(name, P, y_true, results):
    pred = P.argmax(1)
    results.append({"model": name, "accuracy": accuracy_score(y_true, pred),
                    "macro_f1": f1_score(y_true, pred, average="macro", labels=np.unique(y_true)),
                    "weighted_f1": f1_score(y_true, pred, average="weighted"),
                    "top3_acc": topk_acc(P, y_true, 3)})
    return pred

class MarkovModel:
    """P(next | context) with add-alpha smoothing and back-off to shorter contexts."""
    def __init__(self, context_cols, alpha=0.1):
        self.context_cols, self.alpha = context_cols, alpha
    def _key(self, X, cols):
        return X[cols].astype(str).agg("|".join, axis=1) if cols else pd.Series("", index=X.index)
    def fit(self, X, y):
        self.tables = []
        for j in range(len(self.context_cols), -1, -1):          # full context ... empty context
            cols = self.context_cols[:j]
            cnt = pd.crosstab(self._key(X, cols), y).reindex(columns=range(len(CLASSES)), fill_value=0)
            self.tables.append((cols, cnt))
        return self
    def predict_proba(self, X):
        P = np.zeros((len(X), len(CLASSES))); done = np.zeros(len(X), bool)
        for cols, cnt in self.tables:
            key = self._key(X, cols)
            hit = key.isin(cnt.index).to_numpy() & ~done
            if hit.any():
                c = cnt.loc[key[hit]].to_numpy() + self.alpha
                P[hit] = c / c.sum(1, keepdims=True); done |= hit
        return P

Xtr, Xva, Xte = (X.iloc[idx[s]] for s in ["train", "val", "test"])
Xtr_m = Xtr.assign(hour_bin=(Xtr["hour"] // 3).astype(int))
Xva_m = Xva.assign(hour_bin=(Xva["hour"] // 3).astype(int))
Xte_m = Xte.assign(hour_bin=(Xte["hour"] // 3).astype(int))

baselines = {
    "Majority class": MarkovModel([]),
    "Markov-1  P(next|cur)": MarkovModel(["cur_act"]),
    "Markov-2  P(next|prev1,cur)": MarkovModel(["cur_act", "prev1_act"]),
    "Time-aware Markov  P(next|cur,3h-bin)": MarkovModel(["cur_act", "hour_bin"]),
}
res_val, res_test, P_test = [], [], {}
for name, m in baselines.items():
    m.fit(Xtr_m, ytr)
    evaluate(name, m.predict_proba(Xva_m), yva, res_val)
    P_test[name] = m.predict_proba(Xte_m)
    evaluate(name, P_test[name], yte, res_test)
pd.DataFrame(res_val).set_index("model").round(3)

## 4. Temporal gradient-boosting model
`HistGradientBoostingClassifier` uses native categorical splits, handles the mix of categorical
(activity lags, last sensor) and numeric/cyclic time features, and scales to millions of rows.
Early stopping on an internal hold-out keeps it from overfitting. We compare a few settings on the
**validation** period and refit the best one on train+validation before the final test evaluation.

In [ ]:
CAT_COLS = ["cur_act"] + [f"prev{k}_act" for k in range(1, N_LAGS + 1)] + ["last_sensor"]
NUM_COLS = ["hour_sin", "hour_cos", "dow", "is_weekend", "log_duration", "log_n_events", "n_sensors", "log_gap_prev"]
FEATURES = CAT_COLS + NUM_COLS

def fit_encoders(Xf, max_levels=250):
    enc = {}
    for c in CAT_COLS:
        top = Xf[c].astype(str).value_counts().index[:max_levels]   # HGB supports <=255 categories
        enc[c] = {v: i for i, v in enumerate(top)}
    return enc

def encode(Xf, enc):
    Z = Xf[FEATURES].copy()
    for c in CAT_COLS:
        Z[c] = Z[c].astype(str).map(enc[c]).fillna(len(enc[c])).astype(int)   # unseen -> own bucket
    return Z.astype(float)

enc = fit_encoders(Xtr)
Ztr, Zva, Zte = encode(Xtr, enc), encode(Xva, enc), encode(Xte, enc)
cat_mask = np.array([c in CAT_COLS for c in FEATURES])

def full_proba(clf, Z):
    P = np.zeros((len(Z), len(CLASSES)))
    P[:, clf.classes_] = clf.predict_proba(Z)
    return P

grid = [dict(learning_rate=0.1, max_leaf_nodes=31, l2_regularization=0.0),
        dict(learning_rate=0.05, max_leaf_nodes=63, l2_regularization=1.0),
        dict(learning_rate=0.1, max_leaf_nodes=15, l2_regularization=1.0, class_weight="balanced")]
best, best_score = None, -1
for params in grid:
    t0 = time.time()
    clf = HistGradientBoostingClassifier(categorical_features=cat_mask, max_iter=400, early_stopping=True,
                                         validation_fraction=0.1, n_iter_no_change=20,
                                         random_state=RANDOM_STATE, **params).fit(Ztr, ytr)
    P = full_proba(clf, Zva)
    acc, mf1 = accuracy_score(yva, P.argmax(1)), f1_score(yva, P.argmax(1), average="macro", labels=np.unique(yva))
    print(f"{params} -> val acc={acc:.3f} macro-F1={mf1:.3f} ({clf.n_iter_} iters, {time.time()-t0:.1f}s)")
    score = acc + mf1
    if score > best_score:
        best, best_score = params, score
print("Best params:", best)

In [ ]:
# Refit the best configuration on train + validation, then evaluate once on the held-out test period.
trva = np.concatenate([idx["train"], idx["val"]])
enc = fit_encoders(X.iloc[trva])
Ztrva, Zte = encode(X.iloc[trva], enc), encode(Xte, enc)
hgb = HistGradientBoostingClassifier(categorical_features=cat_mask, max_iter=400, early_stopping=True,
                                     validation_fraction=0.1, n_iter_no_change=20,
                                     random_state=RANDOM_STATE, **best).fit(Ztrva, y_all[trva])
P_test["HistGradientBoosting (temporal features)"] = full_proba(hgb, Zte)

# Markov baselines refit on train+val too, for a fair comparison on test
res_test = []
Xtrva_m = X.iloc[trva].assign(hour_bin=(X.iloc[trva]["hour"] // 3).astype(int))
for name, m in baselines.items():
    P_test[name] = m.fit(Xtrva_m, y_all[trva]).predict_proba(Xte_m)
for name, P in P_test.items():
    evaluate(name, P, yte, res_test)
results = pd.DataFrame(res_test).set_index("model").round(3)
results

In [ ]:
ax = results[["accuracy", "macro_f1", "top3_acc"]].plot.barh(figsize=(10, 4.5))
ax.set_title("Next-activity prediction on the held-out test period"); ax.set_xlim(0, 1)
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

## 5. Error analysis of the best model

In [ ]:
best_name = results["macro_f1"].idxmax()
pred = P_test[best_name].argmax(1)
print("Best model on test (macro-F1):", best_name, "\n")
present = np.unique(np.concatenate([yte, pred]))
print(classification_report(yte, pred, labels=present, target_names=CLASSES[present], digits=3, zero_division=0))

In [ ]:
labels_sorted = pd.Series(yte).value_counts().index[:20].to_numpy()
cm = confusion_matrix(yte, pred, labels=labels_sorted, normalize="true")
plt.figure(figsize=(12, 10))
sns.heatmap(cm, xticklabels=CLASSES[labels_sorted], yticklabels=CLASSES[labels_sorted],
            cmap="Blues", annot=len(labels_sorted) <= 15, fmt=".2f")
plt.title(f"Normalised confusion matrix: {best_name}"); plt.xlabel("predicted next"); plt.ylabel("true next"); plt.show()

In [ ]:
n = min(5000, len(Zte))
sub = np.random.RandomState(RANDOM_STATE).choice(len(Zte), n, replace=False)
pi = permutation_importance(hgb, Zte.iloc[sub], y_all[idx["test"]][sub], n_repeats=5,
                            random_state=RANDOM_STATE, scoring="accuracy", n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(figsize=(8, 5), title="Permutation importance (drop in test accuracy)"); plt.show()

In [ ]:
# Accuracy by hour of day: is the next activity easier to predict at some times?
acc_hour = pd.Series(pred == yte).groupby(Xte["hour"].astype(int).to_numpy()).mean()
acc_hour.plot.bar(figsize=(12, 3.5), color="C2", title=f"Test accuracy by hour of day: {best_name}")
plt.ylim(0, 1); plt.xlabel("hour when the current activity ends"); plt.show()

## 6. Save the model and demo inference

In [ ]:
bundle = {"model": hgb, "encoders": enc, "features": FEATURES, "cat_cols": CAT_COLS, "classes": CLASSES}
joblib.dump(bundle, OUT_DIR / "next_activity_hgb.joblib")
results.to_csv(OUT_DIR / "next_activity_results.csv")
print("Saved to", OUT_DIR)

def predict_next(history_episodes, bundle=bundle, k=3):
    """history_episodes: episode DataFrame (home, activity, start, end, n_events, last_sensor,
    n_sensors, duration_min) for one home. Returns the top-k next activities after the last one."""
    h = pd.concat([history_episodes, history_episodes.tail(1)], ignore_index=True)  # dummy row as target
    Xh, _, _ = make_features(h)
    z = encode(Xh.tail(1), bundle["encoders"])
    p = bundle["model"].predict_proba(z)[0]
    top = np.argsort(-p)[:k]
    return [(str(bundle["classes"][bundle["model"].classes_[i]]), round(float(p[i]), 3)) for i in top]

demo_home = meta.loc[idx["test"][0], "home"]
hist = ep[ep["home"] == demo_home].sort_values("start")
cut = int(len(hist) * 0.9)
for i in range(cut, min(cut + 8, len(hist) - 1)):
    h = hist.iloc[: i + 1]
    print(f"{h['end'].iloc[-1]:%a %H:%M}  after {h['activity'].iloc[-1]:<24s} -> predicted {predict_next(h)}"
          f"   | actual: {hist['activity'].iloc[i + 1]}")

## 7. Summary

In [ ]:
b = results.loc[best_name]; mk = results.loc["Markov-1  P(next|cur)"]; mj = results.loc["Majority class"]
print(f"Episodes: {len(ep):,} | classes: {len(CLASSES)} | train+val: {len(trva):,} | test: {len(idx['test']):,}")
print(f"Majority baseline : acc={mj.accuracy:.3f}  macro-F1={mj.macro_f1:.3f}  top-3={mj.top3_acc:.3f}")
print(f"Markov-1          : acc={mk.accuracy:.3f}  macro-F1={mk.macro_f1:.3f}  top-3={mk.top3_acc:.3f}")
print(f"Best ({best_name}): acc={b.accuracy:.3f}  macro-F1={b.macro_f1:.3f}  top-3={b.top3_acc:.3f}")
print("Top-3 most important features:", ", ".join(imp.sort_values(ascending=False).index[:3]))

**Conclusions**

* Representing the stream as **activity episodes** turns a noisy, irregular sensor log into a short
  symbolic sequence per day, on which next-activity prediction is well defined.
* A **1st-order Markov chain** already beats the majority baseline by a wide margin, which confirms
  the strong sequential structure seen in the EDA transition matrix.
* Adding **time of day**, longer history (lags) and episode context (duration, last sensor) in a
  gradient-boosting model improves further, especially top-3 accuracy. That matters for practical uses
  such as prompting or anomaly detection, where a short list of likely next activities is enough.
* Remaining errors are concentrated in activities that are rare, or that share rooms and times of day
  (e.g. different meal types, *Relax* vs *Watch_TV*).
* Possible extensions: sequence models (GRU/Transformer over episodes or raw events), per-home
  fine-tuning, predicting *when* the next activity starts (time-to-event), and cross-home
  evaluation (train on some homes, test on unseen ones).